# LaTeX OCR with Qwen3.5 — as a Nawāt experiment

Image of a handwritten formula in, LaTeX out. This notebook is where you **look**:
at the data before training, and at the results after. The training itself is an
**experiment** — the `vlm-sft` preset, which is Unsloth's Qwen3.5 vision notebook
as a fixed backbone — planned here, approved by you in a terminal, and run by Nawāt.

Why not train in the kernel? A result trained from notebook cells would have none
of an experiment's guarantees: no fixed backbone, no declared parameters, no
approval, no traces. `nawat.begin_run()` refuses for that reason.

### Preflight

Reports what is installed and which GPU this kernel sees. Nothing is installed or
downloaded here.

In [ ]:
import importlib.metadata as meta
import torch

for name in ("nawat", "unsloth", "trl", "transformers", "torch", "vllm"):
    try:
        print(f"{name:13} {meta.version(name)}")
    except meta.PackageNotFoundError:
        print(f"{name:13} not installed")
print("GPU          ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
!nawat check

### Look at the data

`nawat.resolve()` stages the dataset — local cache, then object storage, then
Hugging Face the first time only — and returns an ordinary path. Reading it here
changes nothing.

In [ ]:
import nawat
from datasets import load_dataset

dataset = load_dataset(str(nawat.resolve("datasets/unsloth/LaTeX_OCR")), split="train")
dataset

In [ ]:
dataset[2]["image"]

In [ ]:
from IPython.display import Math, display

display(Math(dataset[2]["text"]))
dataset[2]["text"]

### The experiment

Every block and parameter of the preset, with its default and meaning:

In [ ]:
!nawat preset show vlm-sft

Plan it. Change values with `--set`, never by editing code; the plan prints every
parameter and an approval code. `train.max_steps=30` keeps this first run short.

In [ ]:
!nawat exp plan vlm-sft --name latex-ocr-nb --set train.max_steps=30 --notes "LaTeX OCR from the example notebook"

### Approve it — in a terminal, not here

Approval needs a person at an interactive terminal, by design: an agent (or a
notebook cell) can prepare an experiment but cannot approve it. In a terminal:

```bash
nawat exp approve vlm-sft/latex-ocr-nb
```

Read the table, type the code it shows, then come back and run the next cell.

In [ ]:
!nawat exp run vlm-sft/latex-ocr-nb

### Results

In [ ]:
!nawat exp report vlm-sft/latex-ocr-nb

What the model wrote for the same probe images as training went on:

In [ ]:
!nawat exp traces vlm-sft/latex-ocr-nb --kind samples --limit 3

The loss trace, from the experiment's own metric series:

In [ ]:
import matplotlib.pyplot as plt
from nawat.config import Config
from nawat.lab.experiment import find
from nawat.lab.report import metric_points

points = [p for p in metric_points(find(Config.from_env(), "vlm-sft/latex-ocr-nb")) if "loss" in p and not p.get("event")]
plt.plot([p["step"] for p in points], [p["loss"] for p in points])
plt.xlabel("step"); plt.ylabel("loss"); plt.title("vlm-sft/latex-ocr-nb")
plt.show()

### Next

```bash
nawat exp plan vlm-sft --name latex-ocr-2b --model models/unsloth/Qwen3.5-2B   # a bigger model
nawat exp compare vlm-sft/latex-ocr-nb vlm-sft/latex-ocr-2b                                   # only what differs
nawat sweep plan vlm-sft --vary train.learning_rate=1e-4,2e-4                   # a grid, approved once
```